In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc
import os
import scipy.sparse as sp
import anndata as ad
import pandas as pd
from scipy.spatial import cKDTree
from tqdm import tqdm
from skimage.measure import find_contours

In [2]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/h5ad_data_overlaid/OFM_AD_3_overlaid.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/h5ad_data_overlaid/OFM_AD_3_overlaid.h5ad


In [3]:
batch = adata.obs['batch'][0] 
sample = adata.obs['sample'][0]
adata

/tmp/ipykernel_423571/3043734248.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  batch = adata.obs['batch'][0]
/tmp/ipykernel_423571/3043734248.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample = adata.obs['sample'][0]


AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [4]:
sample_group = adata.obs['age_group'][0]+"_" + adata.obs['disease_status'][0]
sample_group

/tmp/ipykernel_423571/2201400382.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample_group = adata.obs['age_group'][0]+"_" + adata.obs['disease_status'][0]


'Aged_AD'

In [5]:

def tic_normalize(adata, inplace = True, layer_name = None):
    """
    Perform Total Ion Current (TIC) normalization on an AnnData object.
    
    Parameters:
    - adata (AnnData): The input AnnData object with intensity matrix in `.X`.
    - inplace (bool): If True, overwrite `adata.X`. If False, store result in `adata.layers[layer_name]`.
    - layer_name (str): Name of the layer to store normalized matrix. Required if `inplace=False`.

    Returns:
    - None. The AnnData object is modified in-place.
    """

    if not inplace and not layer_name:
        raise ValueError("You must specify 'layer_name' when inplace=False.")

    # Compute TIC per observation
    tic = adata.X.sum(axis=1)
    tic = np.asarray(tic).flatten()
    tic[tic == 0] = 1.0  # Avoid division by zero

    # Normalize
    if sp.issparse(adata.X):
        normalized = adata.X.multiply(1 / tic[:, np.newaxis])
    else:
        normalized = adata.X / tic[:, np.newaxis]

    # Store result
    if inplace:
        adata.layers["raw"] = adata.X.copy()  # Store raw values before normalization
        adata.X = normalized
    else:
        adata.layers[layer_name] = normalized

    # Save TIC to .obs
    adata.obs["tic"] = tic

    print(f"✅ TIC normalization complete. {'Updated adata.X' if inplace else f'Stored in layer: {layer_name}'}")
    

In [6]:
tic_normalize(adata, inplace=True)

✅ TIC normalization complete. Updated adata.X


In [7]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [8]:
foldchange = 3

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=False)

Dropping 666 m/z features: ['136.0150' '137.0251' '138.0287' '139.0359' '154.0255' '155.0350'
 '156.0424' '159.0070' '160.0107' '163.0405' '165.0200' '172.0540'
 '173.0578' '176.0111' '177.0146' '178.0269' '179.0362' '180.0717'
 '181.0163' '185.0686' '187.0399' '189.0757' '190.0676' '191.0379'
 '192.9801' '195.0905' '196.0981' '196.9856' '197.1082' '197.9919'
 '199.0008' '199.0439' '199.9998' '200.0492' '201.0571' '202.0613'
 '202.1793' '203.0369' '203.0680' '203.1800' '204.0398' '210.9897'
 '211.0595' '213.0560' '213.9619' '214.0577' '214.9722' '215.0298'
 '216.0426' '217.0514' '218.0560' '219.0630' '219.9751' '220.0723'
 '220.9800' '226.1111' '227.0378' '228.0455' '229.0025' '229.0488'
 '230.0543' '231.0686' '232.0653' '233.0507' '233.0774' '235.9458'
 '239.0343' '240.0412' '241.0502' '242.0613' '243.0269' '243.0677'
 '244.0353' '245.0458' '246.0516' '247.0594' '248.0624' '249.0743'
 '251.0421' '254.0257' '255.0287' '256.0338' '257.0478' '258.0498'
 '259.0608' '259.2436' '260.0386' '

In [9]:
adata.var_names[0]

'136.0608'

In [10]:
def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        # Global center of mass and distance to tissue centroid
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - tissue_centroid_x) ** 2 + (y_com - tissue_centroid_y) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [11]:
pre_df_metrics, pre_df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 334/334 [00:12<00:00, 27.02it/s]


In [12]:
def mask_low_background_intensities(adata, fold_change=0.2):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [13]:
min_background_intensity = 0.1
adata = mask_low_background_intensities(adata, fold_change=min_background_intensity)

In [14]:

def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):

        # Initialize row for current m/z
        row = {"mz": mz_values[mz_index]}

        # Extract x, y, and intensity for current mz
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()

        # Compute global center of mass
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        # Only tissue pixels
        tissue_mask = adata.obs["tissue"].values == True
        tissue_x = all_x[tissue_mask]
        tissue_y = all_y[tissue_mask]
        tissue_intensities = all_intensities[tissue_mask]

        # Compute weighted tissue centroid (center of mass within tissue)
        x_com_tissue, y_com_tissue = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)

        # Compute distance from global COM to tissue COM
        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - x_com_tissue) ** 2 + (y_com - y_com_tissue) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()

        # Add mean intensities
        row["mean_intensity_tissue"] = np.mean(tissue_intensities) if tissue_intensities.size > 0 else np.nan
        row["mean_intensity_bg"] = np.mean(bg_intensities) if bg_intensities.size > 0 else np.nan

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [26]:
import matplotlib.pyplot as plt

def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(12, 12))  # square figure
    ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=8, label="Background Pixels")
    ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=8, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.8, head_width=0.5)
    
    mz_name = f"{mz_value:.2f}"
    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_name}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.set_aspect("equal")       # maintain equal scale on both axes
    ax.invert_yaxis()            # flip vertically
    plt.tight_layout()
    plt.show()

In [74]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow
from matplotlib.lines import Line2D

def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(8, 6))  # square figure
    bg_scatter = ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=10, label="Background Pixels")
    tissue_scatter = ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=10, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.8, head_width=0.5)

    # Create a dummy arrow for legend
    arrow_legend = FancyArrow(0, 0, 1, 0, color="gray", alpha=1, width=0.1)

    # Custom legend handles
    legend_handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Background Pixels'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Nearest Tissue Border'),
        Line2D([0], [0], color='gray', lw=2, label='Direction (BG → Tissue)')
    ]

    mz_name = f"{mz_value:.2f}"
    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_name}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(handles=legend_handles)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


In [16]:
df_metrics, df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 334/334 [00:09<00:00, 34.52it/s]


In [75]:
# Visualize arrows for m/z=885.55 (example)
visualize_distance_map(df_border, mz_value=651.5335, n_arrows=100)


In [ ]:

def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot"):
    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
    bg_point = (max_row["bg_x"], max_row["bg_y"])
    nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap with square pixels
    plt.scatter(x, y, c=intensities, s=11, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot center of mass
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")

    # Max distance point and nearest tissue point
    plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
             color="orange", linestyle="--", label="Max dist to tissue")
    plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
    plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue margin line
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])
#custom_cmap = "binary"
visualize_mz_feature(adata, df_metrics, df_border, mz_value=float(adata.var_names[0]), cmap=custom_cmap)

In [ ]:
def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot", save_path=None):
    from skimage.measure import find_contours
    import matplotlib.pyplot as plt
    import numpy as np

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    if df_mz_border.empty:
        print(f"⚠️ Skipping m/z {mz_value:.4f} — no matching metric data found.")
    else:
        max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
        bg_point = (max_row["bg_x"], max_row["bg_y"])
        nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap
    plt.scatter(x, y, c=intensities, s=12, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot COMs and max distance markers
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")
    if not df_mz_border.empty:
        plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
                color="orange", linestyle="--", label="Max dist to tissue")
        plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
        plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue contour
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()


In [ ]:
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all
0,104.1083,0.600647,0.000704,0.000022,32.202484,6.017918,492,76.412290,57.755337,76.605761,58.323973
1,129.1380,0.975817,0.000784,0.000049,27.730849,4.213910,856,71.095270,55.197670,71.525767,56.073393
2,136.0607,0.963061,0.001809,0.000178,31.112698,8.229154,1994,70.442155,54.744987,70.381059,55.706108
3,161.1337,2.824843,0.000672,0.000051,34.713110,10.048400,1280,71.013363,58.227767,70.852013,61.047998
4,184.0718,0.180025,0.015142,0.000743,31.064449,3.921431,787,71.329455,56.407549,71.348089,56.228491
...,...,...,...,...,...,...,...,...,...,...,...
417,1544.1532,0.526122,0.000353,0.000014,30.805844,2.156259,416,70.077343,54.540065,70.353020,54.091951
418,1545.1484,0.575210,0.000237,0.000008,31.144823,1.906250,310,70.230876,54.623610,70.479784,54.105044
419,1548.1879,0.466076,0.000233,0.000008,25.000000,1.664409,258,70.156770,58.817172,70.498127,58.499837
420,1560.1166,0.168349,0.000263,0.000003,22.203603,1.698974,160,70.987305,56.703492,71.096688,56.575520


In [ ]:
def score_mz_features(df_metrics, area_weight=0.95):
    # Normalize metrics
    # Normalize distance metrics
    df_metrics["normalized_bg_dist_mean"] = (
        df_metrics["bg_dist_mean"] - df_metrics["bg_dist_mean"].min()
    ) / (df_metrics["bg_dist_mean"].max() - df_metrics["bg_dist_mean"].min())

    # Normalize area metrics
    df_metrics["normalized_bg_nonzero_area"] = (
        df_metrics["bg_nonzero_area"] - df_metrics["bg_nonzero_area"].min()
    ) / (df_metrics["bg_nonzero_area"].max() - df_metrics["bg_nonzero_area"].min())

    # Calculate dispersion score
    # The dispersion score is a weighted combination of the normalized metrics
    # Higher score means better dispersion (lower distance, higher area)
    # Note: area_weight controls the balance between area and distance
    df_metrics["dispersion_score"] = (
        area_weight * df_metrics["normalized_bg_nonzero_area"]
        + (1 - area_weight) * df_metrics["normalized_bg_dist_mean"]
    )
    
    return df_metrics


In [ ]:
area_weight = 0.95
# Score m/z features based on dispersion metrics
# This will add a new column 'dispersion_score' to df_metrics
df_metrics = score_mz_features(df_metrics, area_weight=area_weight)
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all,normalized_bg_dist_mean,normalized_bg_nonzero_area,dispersion_score
0,104.1083,0.600647,0.000704,0.000022,32.202484,6.017918,492,76.412290,57.755337,76.605761,58.323973,0.450842,0.130761,0.146765
1,129.1380,0.975817,0.000784,0.000049,27.730849,4.213910,856,71.095270,55.197670,71.525767,56.073393,0.288758,0.228296,0.231319
2,136.0607,0.963061,0.001809,0.000178,31.112698,8.229154,1994,70.442155,54.744987,70.381059,55.706108,0.649514,0.533226,0.539041
3,161.1337,2.824843,0.000672,0.000051,34.713110,10.048400,1280,71.013363,58.227767,70.852013,61.047998,0.812966,0.341908,0.365461
4,184.0718,0.180025,0.015142,0.000743,31.064449,3.921431,787,71.329455,56.407549,71.348089,56.228491,0.262480,0.209807,0.212441
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,1544.1532,0.526122,0.000353,0.000014,30.805844,2.156259,416,70.077343,54.540065,70.353020,54.091951,0.103886,0.110397,0.110071
418,1545.1484,0.575210,0.000237,0.000008,31.144823,1.906250,310,70.230876,54.623610,70.479784,54.105044,0.081423,0.081994,0.081965
419,1548.1879,0.466076,0.000233,0.000008,25.000000,1.664409,258,70.156770,58.817172,70.498127,58.499837,0.059695,0.068060,0.067642
420,1560.1166,0.168349,0.000263,0.000003,22.203603,1.698974,160,70.987305,56.703492,71.096688,56.575520,0.062800,0.041801,0.042851


In [ ]:
df_metrics_filtered = df_metrics[["mz", "dispersion_score", "bg_nonzero_area","normalized_bg_nonzero_area", "bg_dist_mean", "normalized_bg_dist_mean","mean_intensity_tissue", "mean_intensity_bg"]]
df_metrics_filtered.to_csv(f"/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/mz_metrics_{sample_group}_{sample}.csv", index=False)

In [ ]:
output_folder = f"/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/{sample_group}_{sample}_output_{min_background_intensity}_score_{area_weight}"

if os.path.exists(output_folder):
    print(f"⚠️ Warning: Output folder '{output_folder}' already exists. Overwriting.")
else:
    os.makedirs(output_folder, exist_ok=True)

df_sorted = df_metrics.sort_values(by="dispersion_score", ascending=False)
rank_num = 1

# Visualize and save m/z features based on dispersion score
# Loop through sorted m/z values and save visualizations
for mz in df_sorted["mz"]:
    filename = f"{rank_num}_mz_{mz:.4f}.png"
    filepath = os.path.join(output_folder, filename)
    
    custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
        (0.0, "#454545"),   # white
        (0.00000001, "#000000"),  # black
        (0.10, "#000080"),  # navy
        (0.15, "#0000FF"),  # blue
        (0.30, "#8000FF"),  # purple-ish
        (0.45, "#FF0000"),  # red
        (0.60, "#FF8000"),  # orange
        (0.75, "#FFFF00"),  # yellow
        (1.0, "#FFFFFF")   # white
    ])
    
    visualize_mz_feature(adata, df_metrics, df_border, mz_value=mz, cmap=custom_cmap, save_path=filepath)
    print(f"Saved: {filepath}")
    rank_num += 1

Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/1_mz_734.5667.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/2_mz_350.0835.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/3_mz_735.5729.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/4_mz_736.5680.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/5_mz_761.5907.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/6_mz_598.9324.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score_0.95/7_mz_732.6036.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/Young_AD_3-6_output_0.1_score